In [1]:
import os
import json
from sentence_transformers import SentenceTransformer
from chromadb.config import Settings
import chromadb

# Initialize embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Setup Chroma client with persistence directory
client = chromadb.Client(Settings(
    persist_directory="./chroma_db",  # local folder to store DB files persistently
    anonymized_telemetry=False
))

collection_name = "my_study_docs"

# Create or get the existing collection
if collection_name in [c.name for c in client.list_collections()]:
    collection = client.get_collection(collection_name)
else:
    collection = client.create_collection(name=collection_name)

PREPROCESSED_FOLDER = "preprocessed"

def load_chunks_from_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    # Your example JSON has keys "file_path" and "chunks"
    return data.get("file_path", os.path.basename(file_path)), data.get("chunks", [])

def embed_and_add_to_chroma(file_path, chunks):
    # Generate embeddings for all chunks at once (batch)
    embeddings = model.encode(chunks, show_progress_bar=True)
    # Unique IDs per chunk combining filename and index
    ids = [f"{file_path}_{i}" for i in range(len(chunks))]
    # Metadata per chunk
    metadatas = [{"source_file": file_path, "chunk_index": i} for i in range(len(chunks))]
    # Raw text documents are just the chunks themselves
    documents = chunks

    # Add to Chroma collection
    collection.add(
        ids=ids,
        embeddings=embeddings.tolist() if hasattr(embeddings, "tolist") else embeddings,
        metadatas=metadatas,
        documents=documents,
    )

def main():
    for filename in os.listdir(PREPROCESSED_FOLDER):
        if not filename.endswith(".json"):
            continue
        full_path = os.path.join(PREPROCESSED_FOLDER, filename)
        file_path, chunks = load_chunks_from_json(full_path)
        if not chunks:
            print(f"No chunks found in {filename}, skipping.")
            continue
        print(f"Embedding and adding {len(chunks)} chunks from {file_path}...")
        embed_and_add_to_chroma(file_path, chunks)

    # Persist the collection to disk
    collection.persist()
    print("Chroma DB persisted to disk.")

if __name__ == "__main__":
    main()


/home/hazem/dev/Assist/glob_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RuntimeError: [91mYour system has an unsupported version of sqlite3. Chroma                     requires sqlite3 >= 3.35.0.[0m
[94mPlease visit                     https://docs.trychroma.com/troubleshooting#sqlite to learn how                     to upgrade.[0m

In [ ]:
import _sqlite3
_sqlite3.__file__

'/home/hazem/.pyenv/versions/3.10.12/lib/python3.10/lib-dynload/_sqlite3.cpython-310-x86_64-linux-gnu.so'

In [7]:
%pip freeze | grep sql

pysqlite3==0.5.4
pysqlite3-binary==0.5.4
Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys 

In [6]:
for i in sys.modules:
    if "sqlite" in i:
        print(i, sys.modules[i])

_sqlite3 <module '_sqlite3' from '/home/hazem/.pyenv/versions/3.10.12/lib/python3.10/lib-dynload/_sqlite3.cpython-310-x86_64-linux-gnu.so'>
sqlite3.dbapi2 <module 'sqlite3.dbapi2' from '/home/hazem/.pyenv/versions/3.10.12/lib/python3.10/sqlite3/dbapi2.py'>
sqlite3 <module 'sqlite3' from '/home/hazem/.pyenv/versions/3.10.12/lib/python3.10/sqlite3/__init__.py'>


In [ ]:
import sqlite3
sqlite3.sqlite_version

: 